Cell 1: Setup & Environment Override

In [1]:
import os
import sys
import ssl
import requests
import pandas as pd
import numpy as np
import faiss
import pickle
from sentence_transformers import SentenceTransformer
from requests.packages.urllib3.exceptions import InsecureRequestWarning

# --- NUCLEAR SSL BYPASS (S&P Global / Corporate Proxy Fix) ---

# 1. Disable Python's default SSL check
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

# 2. Disable Hugging Face & Requests checks
os.environ['CURL_CA_BUNDLE'] = ''
os.environ['HF_HUB_DISABLE_SSL_VERIFY'] = '1'
requests.packages.urllib3.disable_warnings(InsecureRequestWarning)

# 3. SAFE MONKEY PATCH (Prevents Recursion Error)
# We check if we already saved the original function to avoid overwriting it on re-runs
if not hasattr(requests.Session, '_original_merge_environment_settings'):
    requests.Session._original_merge_environment_settings = requests.Session.merge_environment_settings

def merge_environment_settings(self, url, proxies, stream, verify, cert):
    # Always call the ORIGINAL function with verify=False
    return self._original_merge_environment_settings(url, proxies, stream, False, cert)

requests.Session.merge_environment_settings = merge_environment_settings

# 4. Add 'src' to path
sys.path.append(os.path.abspath('..'))

print("✅ Environment Configured: SSL Verification Disabled (Safe Mode)")

c:\Users\ASHUTOSH_SHARMA2\OneDrive - S&P Global\Desktop\Projects\semantic-commodity-enricher\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Environment Configured: SSL Verification Disabled (Safe Mode)


Load the Vector Index & Metadata

In [ ]:
# Configuration
INDEX_PATH = "../data/output/vector_store.index"
META_PATH = "../data/output/vector_store_meta.pkl"
MODEL_NAME = "sentence-transformers/all-MiniLM-L12-v2"

def load_system():
    if not os.path.exists(INDEX_PATH):
        raise FileNotFoundError(" FAISS index not found. Run main.py first.")

    index = faiss.read_index(INDEX_PATH)

    with open(META_PATH, "rb") as f:
        metadata = pickle.load(f)

    print("Loading transformer model…")
    model = SentenceTransformer(MODEL_NAME)

    return index, metadata, model

index, metadata, model = load_system()
print(f" System Loaded: {index.ntotal} commodities indexed.")

Loading transformer model…
 System Loaded: 10 commodities indexed.


Define the Search Logic

In [7]:
def find_commodity(user_query, k=3):
    # Encode & normalize for cosine similarity
    vec = model.encode([user_query], normalize_embeddings=True)
    vec = np.array(vec, dtype="float32")

    # FAISS search (IndexFlatIP)
    scores, indices = index.search(vec, k)

    results = []
    for rank, idx in enumerate(indices[0]):
        if idx == -1:
            continue

        meta = metadata[idx]

        results.append({
            "query": user_query,
            "match": meta["label"],
            "qid": meta["qid"],
            "score": float(scores[0][rank])
        })

    return results


The "Trader Chat" Simulation (The Demo)

In [6]:
# Real-world messy queries
queries = [
    "Need price for North Sea oil",
    "Texas light sweet crude quote",
    "Henry Hub gas spot",
    "Gold bullion price",
    "sour crude price benchmark dubai"
]

rows = []
for q in queries:
    match = find_commodity(q, k=1)[0]
    rows.append(match)

df = pd.DataFrame(rows)
df


,user_query,match_label,canonical_qid,distance
0,Need price for North Sea oil,Dubai Crude,Q4037659,1.061599
1,Texas light sweet crude quote,Brent Crude,Q1270246,1.180839
2,Henry Hub gas spot,natural gas,Q40858,1.490550
3,Gold bullion price,Brent Crude,Q1270246,1.325021
4,sour crude price benchmark dubai,Dubai Crude,Q4037659,0.612369


Inspecting the Graph

In [24]:
from rdflib import Graph, Namespace

g = Graph()

# Load your commodities
g.parse("../data/output/commodities.ttl", format="turtle")

# Load QUDT (this MUST exist)
g.parse("../data/cache/qudt-units.ttl", format="turtle")

print(f"Graph Loaded: {len(g):,} triples")


Graph Loaded: 60,931 triples


In [25]:
def get_conversion_factor(commodity_name):
    query = """
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX qudt: <http://qudt.org/schema/qudt/>

    SELECT ?unit ?mult
    WHERE {
        ?c rdfs:label ?name .
        FILTER(?name = "%s")
        ?c qudt:unit ?unit .
        OPTIONAL { ?unit qudt:conversionMultiplier ?mult . }
    }
    """ % commodity_name

    results = list(g.query(query))
    if not results:
        return None

    unit_uri, factor = results[0]
    return {
        "unit": unit_uri,
        "conversion_factor": float(factor) if factor else None
    }


In [26]:
# The New Way: Ask the Graph
# We don't know the multiplier. We don't care. The ontology knows.
get_conversion_factor("Brent Crude")


{'unit': rdflib.term.URIRef('http://qudt.org/vocab/unit/BBL'),
 'conversion_factor': 0.1589873}

In [29]:
#Convert Commodity into a new unit automatically
def convert_volume(value, commodity_name, target_unit_uri):
    info = get_conversion_factor(commodity_name)
    if not info or not info["conversion_factor"]:
        raise ValueError("No conversion multiplier available")

    source_factor = info["conversion_factor"]

    # Look up target-unit multiplier from QUDT
    target_query = """
    PREFIX qudt: <http://qudt.org/schema/qudt/>
    SELECT ?mult WHERE { <%s> qudt:conversionMultiplier ?mult . }
    """ % target_unit_uri

    target_result = list(g.query(target_query))
    target_factor = float(target_result[0][0])

    return value * (source_factor / target_factor)


In [30]:
# Convert 1 Barrel of Brent → m3
convert_volume(1, "Brent Crude", "http://qudt.org/vocab/unit/M3")

0.1589873